# A code for Data Engineering to merge Market Prices and Market Arrivals into a single "Master" file

In [ ]:
import pandas as pd
import warnings
import numpy as np
import os

warnings.filterwarnings('ignore')

def execute_market_merge_v4():
    print(">>> INITIALIZING PHASE 1: NUMERIC ENFORCEMENT...")
    

    raw_data_path = os.path.join('..', 'Data', 'Raw')
    processed_data_path = os.path.join('..', 'Data', 'Processed')
    
    # Ensure the Processed directory exists before saving
    os.makedirs(processed_data_path, exist_ok=True)

    try:
        prices_path = os.path.join(raw_data_path, 'Punjab_Market_Data_Raw_Final.xlsx')
        arrivals_path = os.path.join(raw_data_path, 'Punjab_Arrivals_Resurrected.csv')

        prices_df = pd.read_excel(prices_path)
        arrivals_df = pd.read_csv(arrivals_path)
    except Exception as e:
        print(f"FAILED TO LOAD FILES: {e}")
        return

    # --- 1. CLEAN PRICES  ---
    # Strip commas/spaces and force to numeric, turning errors to NaN
    prices_df['Max_Price'] = pd.to_numeric(
        prices_df['Max_Price'].astype(str).str.replace(',', '').str.strip(), 
        errors='coerce'
    )
    
    # Drop rows where Price is now NaN (meaning it was garbage text)
    prices_df = prices_df.dropna(subset=['Max_Price'])

    # --- 2. FUZZY COMMODITY MAPPING ---
    def map_commodity(name):
        name = str(name).upper()
        if 'POTATO' in name: return 'Potato'
        if 'ONION' in name: return 'Onion'
        if 'TOMATO' in name: return 'Tomato'
        if 'WHEAT' in name: return 'Wheat'
        if 'BANANA' in name: return 'Banana'
        return None

    prices_df['Normalized_Comm'] = prices_df['Agri_Commodity_Name'].apply(map_commodity)
    prices_filtered = prices_df.dropna(subset=['Normalized_Comm']).copy()
    
    print(f"DEBUG: Valid Price Rows: {len(prices_filtered)}")

    # --- 3. STANDARDIZE KEYS ---
    prices_filtered['City'] = prices_filtered['City'].astype(str).str.strip().str.upper()
    prices_filtered['Date'] = pd.to_datetime(prices_filtered['Date']).dt.normalize()
    
    arrivals_df['City'] = arrivals_df['City'].astype(str).str.strip().str.upper()
    arrivals_df['Date'] = pd.to_datetime(arrivals_df['Date']).dt.normalize()
    arrivals_df = arrivals_df.loc[:, ~arrivals_df.columns.str.contains('^Unnamed')]

    # --- 4. AGGREGATE & PIVOT---
    # We specify aggfunc='max' so it doesn't default to 'mean'
    price_wide = prices_filtered.pivot_table(
        index=['Date', 'City'], 
        columns='Normalized_Comm', 
        values='Max_Price',
        aggfunc='max' 
    ).reset_index()

    price_wide.rename(columns={
        'Potato': 'Potato_Price', 
        'Tomato': 'Tomato_Price', 
        'Onion': 'Onion_Price',
        'Wheat': 'Wheat_Price',
        'Banana': 'Banana_Price'
    }, inplace=True)

    # --- 5. THE MASTER MERGE ---
    master_market_df = pd.merge(price_wide, arrivals_df, on=['Date', 'City'], how='inner')

    if not master_market_df.empty:
        # Save output dynamically to the Processed folder
        output_file = os.path.join(processed_data_path, "Master_Market_Fact.csv")
        master_market_df.to_csv(output_file, index=False)
        print("\n" + "="*50)
        print(f"SUCCESS: {len(master_market_df)} Rows Harmonized.")
        print(f"File Saved: {output_file}")
        print("="*50)
    else:
        print("\n!!! NO MATCHING DATE/CITY PAIRS FOUND !!!")
        print(f"Price Cities: {price_wide['City'].unique()[:3]}")
        print(f"Arrival Cities: {arrivals_df['City'].unique()[:3]}")

if __name__ == "__main__":
    execute_market_merge_v4()

>>> INITIALIZING PHASE 1: NUMERIC ENFORCEMENT...
DEBUG: Valid Price Rows: 645

SUCCESS: 93 Rows Harmonized.
File Saved: PCCMD_Master_Market_Fact_Banana_Eecheeya.csv
